# Sentence Pairs: Find sentences that are similar to eachother

This notebook uses the cosine similarity scores to find sentences that appear to be the most similar to eachother.
- Creates a dataframe with sentence pairs.
- Look into the sentences with the highest similarity scores.
- Distribution of highest similarity scores.
- Check sentences with a low 'highest similarity score'. (These sentences are considered to be 'unique' in the corpus, given that they did not manage to get a high match with another sentence in the corpus.)

### Settings

In [ ]:
# Settings
similarity_file = "similarity_corpus_free_3600_250606.pickle"
raw_corpus_file = "corpus_free_3600_250606.csv"

## Initialisation

### Imports

In [ ]:
# import
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Load vectors and corpus

In [ ]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

In [ ]:
# Load cosine similarity scores
with open(f"../../data/vectors/{similarity_file}", 'rb') as handle:
    similarities = pickle.load(handle)

In [ ]:
# sample
df = pd.DataFrame(similarities)
df.iloc[:5, :10]

Similarity scores are correctly loaded if row and column indices with the same number == `1`

## Visualise with clustermap

(Just a small part. Ploting all data is too much.)

In [ ]:
# plot clustermap
subset = 300 #158
sns.clustermap(similarities[:subset, :subset])
plt.title(f"Sentence similarty (cosine) on a subset ({subset}) of corpus")
plt.show()

The two large clusters here might be papers?

## Explore sentence pairs

### Create a data frame with sentence pairs

In [ ]:
# sample
df = pd.DataFrame(similarities)
df.iloc[:5, :10]

In [ ]:
# Substract 2 from the cosine similarity score that matched itself, so that it turns the value to -1.
retrieve_sim_sent = np.subtract(similarities, np.identity(similarities.shape[0]) *2)

pd.DataFrame(retrieve_sim_sent).head()

In [ ]:
# Retrieve index of sentence that is most similar to sentence at array position.
# Example: [1, 0, 5, ...] -> 0st sentence matches best to 1nd sentence, 1nd " 0st, 3rd " 5th.
sent_pairs = retrieve_sim_sent.numpy().argmax(axis=0)
sent_pairs

In [ ]:
# Retrieve the cos_sim score for the pairs
sent_pairs_score = np.array([df.loc[i_r, i_c] for i_r, i_c in enumerate(sent_pairs)])
sent_pairs_score

In [ ]:
# Gather info into one df
df_sent_pairs = pd.DataFrame(
    {
        # 'i_r': np.arange(0, sent_pairs.shape[0]),
        'i_c': sent_pairs, # sentence_id of cos_sim with highest score.
        'cos_sim': sent_pairs_score # Highest cos_sim score for sentence.
    }
)

# Add paper_name or row (= pmid)
df_sent_pairs = pd.merge(df_sent_pairs, df_corpus[['paper_name','sentence_text']], left_index=True, right_index=True)
# Add paper_name or column (= pmid)
df_sent_pairs = pd.merge(df_sent_pairs, df_corpus[['paper_name','sentence_text']], left_on='i_c', right_index=True)
# Rename the just added columns
df_sent_pairs = df_sent_pairs.rename(columns= {'paper_name_x': 'i_r_pmid',
                                               'paper_name_y': 'i_c_pmid',
                                               'sentence_text_x': 'i_r_text',
                                                'sentence_text_y': 'i_c_text',
                                               })

df_sent_pairs.head(10)

### Distribution of highest similarity scores

In [ ]:
# Distribution of highest cosine similarity score for a sentence
df_sent_pairs['cos_sim'].hist(bins= np.arange(0,1.2,0.05))

plt.xlabel('cosine similarity score')
plt.ylabel('absolute number of sentences')

plt.vlines(0.75, 0, 4000, color='orange')

plt.title('Distribution of highest cosine similarity score for a sentence')
plt.show()

In [ ]:
# median
df_sent_pairs['cos_sim'].median()

In [ ]:
# describe
df_sent_pairs['cos_sim'].describe()

This follows a normal distribution. It shows that if we place a threshold around the mean (= ~0.75), it will exclude ~50% of the sentences.

Note that there are a few sentences that are 1=<, suggesting sentences to be duplicates.

See next section for further break down.

### Investigate sentence pairs

Note that there are a few sentences that are 1=<, suggesting sentences to be duplicates.

In [ ]:
# How many sentences have cos_sim of 1 or higher?
# Number of sentences that appear to be exactly the same:
print(f"Number of sentences that appear to be exactly the same: {df_sent_pairs[df_sent_pairs['cos_sim'] >= 1].shape[0]}")

In [ ]:
# Show samples
df_sent_pairs[df_sent_pairs['cos_sim'] >= 1].sample(10)

Possibly this is explained by papers using the same header multiple times in different sections. Duplicates also contain standard disclaimers (e.g. 'The authors declare no competing interests.', etc.).


In [ ]:
# Look at sentence pairs with the highest similarity
df_sent_pairs.sort_values('cos_sim', ascending= False).head(20)

(Interesting to report!!) The first three sentences appear in three different papers, from the same authors. pmid = [11826108, 12598611, 12867501].

Then there are sentences that are 'standard disclaimers'. (i.e. 'The authors declare no conflict of interest.', 'Written informed consent was obtained from...', etc.)
Then there are sentences that are used as headers. Like for sentence_id 16262 and 15773.


##### 'Worst' matches

Check if sentences with a low similarity score do not seem similar. These sentences are considered to be 'unique' in the corpus, given that they did not manage to get a high match with another sentence in the corpus.

In [ ]:
# Sentences with the lowest 'highest similarity match'
df_sent_pairs.sort_values('cos_sim', ascending= True).head(10)

In [ ]:
# Get full sentences
df_sent_pairs.sort_values('cos_sim', ascending= True).head(5).values

To me these sentences already appear seem somewhat similar. (the first is about 'Transcranial B-mode sonography monitors', the second about knee/joint injury and swellings)

### Sample Time!

Sample through the matches and read the full text.

In [ ]:
# Sample pairs from the df
i_sample = df_sent_pairs.sample(10).index.to_numpy()
df_sent_pairs.loc[i_sample]

In [ ]:
# Print the metrics and sentences
for i in i_sample:
    print(f"\ni_c: {df_sent_pairs.loc[i, 'i_c']} | i_r: {df_sent_pairs.loc[i].name} | Cosine similairty score: {df_sent_pairs.loc[i, 'cos_sim']}")
    print(f"Sentence row:    {df_sent_pairs.loc[i, 'i_r_text']}")
    print(f"Sentence column: {df_sent_pairs.loc[i, 'i_c_text']}")
    print("---")